In [ ]:
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
import faiss
import pickle

dataset = load_dataset("hoatac/Nutuk_QA")["train"]

chunks = [item["question"] + " " + item["answer"] for item in dataset]

print(f"Toplam chunk sayısı: {len(chunks)}")
print(f"İlk chunk örneği:\n{chunks[0]}")


Toplam chunk sayısı: 6396
İlk chunk örneği:
"Milli maksatlan fiil mevkiine ulaştıracak kafi vasıtalar bulamadığımdan" ifadesi ne anlama geliyor? Bu ifade, milli hedeflere ulaşmak için yeterli araçların bulunamadığını belirtiyor. Yani, mevcut şartlar altında milli amaçlara ulaşmanın mümkün olmadığı düşünülüyor.


In [4]:
embedder = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

embeddings = embedder.encode(chunks, convert_to_numpy=True, show_progress_bar=True)
print(f"Embedding matrisi şekli: {embeddings.shape}")

index = faiss.IndexFlatL2(embeddings.shape[1])
index.add(embeddings)
print(f"Index toplam vektör sayısı: {index.ntotal}")

faiss.write_index(index, "nutuk.index")

with open("nutuk_chunks.pkl", "wb") as f:
    pickle.dump(chunks, f)


Batches:   0%|          | 0/200 [00:00<?, ?it/s]

Embedding matrisi şekli: (6396, 384)
Index toplam vektör sayısı: 6396


In [5]:
embedder = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
index = faiss.read_index("nutuk.index")

with open("nutuk_chunks.pkl", "rb") as f:
    chunks = pickle.load(f)

def retrieve_chunks(question, top_k=20):
    question_embedding = embedder.encode([question])
    distances, indices = index.search(question_embedding, top_k)
    result_chunks = []
    for idx in indices[0]:
        if idx == -1 or idx >= len(chunks):
            continue  # geçersiz indeksleri atla
        result_chunks.append(chunks[idx])
    return result_chunks


In [6]:
from transformers import pipeline

model_name = "savasy/bert-base-turkish-squad"  
qa_pipeline = pipeline("question-answering", model=model_name, tokenizer=model_name, max_answer_len=100)


Some weights of the model checkpoint at savasy/bert-base-turkish-squad were not used when initializing BertForQuestionAnswering: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForQuestionAnswering from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForQuestionAnswering from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cuda:0


In [9]:
def answer_question_bert(question, top_k=10):
    context = "\n".join(retrieve_chunks(question, top_k=top_k))
   
    result = qa_pipeline(question=question, context=context)
    
    return result.get('answer', 'Cevap bulunamadı.')

print(answer_question_bert("Atatürk Samsun'a ne zaman çıkıştır?"))


19 Mayıs 1919'da


In [8]:
def answer_question_bert(question, top_k=10):
    # 1️⃣ Nutuk'tan en ilgili pasajları getir
    chunks = retrieve_chunks(question, top_k=top_k)
    context = "\n".join(chunks)

    # 2️⃣ QA pipeline ile soruyu cevapla
    result = qa_pipeline(question=question, context=context)
    
    # 3️⃣ Sadece cevabı döndür
    return result.get('answer', 'Cevap bulunamadı.')

# Örnek kullanım
print(answer_question_bert("Türkiye için Atatürk neler yaptı?", top_k=10))


ordunun insan ve nakliye vasıtaları bakımından kuvvetini artırmaya, iaşesini ve giydirilmesini temin ve tanzime yönelik tedbirler ve tertibatlar almıştır.
